# Day 9 v1 — Silver Transformation: Charging Sessions (Blob CSV → Silver Delta)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions/` (Delta)

This is the **learning version** — one entity (charging_sessions), every step written explicitly.
Same teaching pattern as Day 8 v1 but for Blob CSV data instead of API JSON.

**Blob CSV Bronze structure:**
```
bronze-volume/realtime/charging_sessions/YYYY/MM/DD/HH/sessions_YYYYMMDD_HHMM.csv
```

**Steps:**
1. Read all Bronze CSV files for charging_sessions
2. Cast columns to correct types
3. Trim whitespace from string columns
4. Add Silver audit columns
5. Deduplicate on `session_id`
6. Write to Silver as Delta table (full overwrite)
7. Verify the output

In [ ]:
# Cell 1 — Imports + clear all cached state from previous runs
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone

# Clear any cached dataframes/tables from prior runs in this session
spark.catalog.clearCache()

# Drop any stale temp views
for v in spark.catalog.listTables("default"):
    if v.isTemporary:
        spark.catalog.dropTempView(v.name)

print("Imports OK — cache cleared")

In [0]:
display(dbutils.fs.ls("/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/01/06"))

path,name,size,modificationTime
dbfs:/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/2026/06/01/06/sessions_20260601_0600.csv,sessions_20260601_0600.csv,10773225,1783745990000


In [0]:
# Cell 2 — Constants
BRONZE_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions"
SILVER_PATH = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_PATH}")
print(f"Run time    : {RUN_TS}")

Bronze path : /Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions
Silver path : /Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions
Run time    : 2026-07-16T08:55:19Z


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType

CS_SCHEMA = StructType([
    StructField("session_id",          StringType(), True),
    StructField("charger_id",          StringType(), True),
    StructField("station_id",          StringType(), True),
    StructField("vehicle_id",          StringType(), True),
    StructField("customer_id",         StringType(), True),
    StructField("plug_in_ts",          StringType(), True),
    StructField("charge_end_ts",       StringType(), True),
    StructField("duration_min",        StringType(), True),
    StructField("energy_kwh",          StringType(), True),
    StructField("peak_kw",             StringType(), True),
    StructField("connector_type",      StringType(), True),
    StructField("session_status",      StringType(), True),
    StructField("tariff_id",           StringType(), True),
    StructField("raw_device_temp_c",   StringType(), True),
    StructField("signal_strength_dbm", StringType(), True),
    StructField("firmware_ver",        StringType(), True),
    StructField("state_code",          StringType(), True),
    StructField("protocol_version",    StringType(), True),
    StructField("ingested_at",         StringType(), True),
])
CS_CSV_COLS = [f.name for f in CS_SCHEMA.fields]

cs_raw = (
    spark.read
    .option("header", "true")
    .option("recursiveFileLookup", "true")
    .schema(CS_SCHEMA)
    .csv(BRONZE_PATH)
    .select(
        *[F.col(c) for c in CS_CSV_COLS],
        F.col("_metadata.file_path").alias("_file_path")
    )
)

cs_raw = (
    cs_raw
    .withColumn("_year",  F.regexp_extract(F.col("_file_path"), r"/(\d{4})/", 1))
    .withColumn("_month", F.regexp_extract(F.col("_file_path"), r"/\d{4}/(\d{2})/", 1))
    .withColumn("_day",   F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/(\d{2})/", 1))
    .withColumn("_hour",  F.regexp_extract(F.col("_file_path"), r"/\d{4}/\d{2}/\d{2}/(\d{2})/", 1))
    .withColumn("_upd_str", F.concat_ws(" ", F.concat_ws("-", F.col("_year"), F.col("_month"), F.col("_day")), F.col("_hour")))
    .drop("_file_path", "_year", "_month", "_day", "_hour")
)

typed_df = cs_raw.select(
    F.col("session_id").cast("string"),
    F.col("charger_id").cast("string"),
    F.col("station_id").cast("string"),
    F.col("vehicle_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("plug_in_ts").cast("timestamp"),
    F.col("charge_end_ts").cast("timestamp"),
    F.col("duration_min").cast("integer"),
    F.col("energy_kwh").cast("decimal(10,4)"),
    F.col("peak_kw").cast("decimal(10,4)"),
    F.col("connector_type").cast("string"),
    F.col("session_status").cast("string"),
    F.col("tariff_id").cast("string"),
    F.col("raw_device_temp_c").cast("decimal(6,2)"),
    F.col("signal_strength_dbm").cast("integer"),
    F.col("firmware_ver").cast("string"),
    F.col("state_code").cast("string"),
    F.col("protocol_version").cast("string"),
    F.col("ingested_at").cast("timestamp"),
    F.to_timestamp(F.col("_upd_str"), "yyyy-MM-dd HH").alias("updated_at"),
)

print(f"Row count : {typed_df.count()}")
typed_df.printSchema()
typed_df.show(3, truncate=True)

In [0]:
# Cell 5 — Trim whitespace from all string columns
# CSV files can have trailing spaces in string fields.

string_cols = [c for c, t in typed_df.dtypes if t == "string"]
trimmed_df  = typed_df
for col in string_cols:
    trimmed_df = trimmed_df.withColumn(col, F.trim(F.col(col)))

print(f"Trimmed string columns: {string_cols}")

Trimmed string columns: ['session_id', 'charger_id', 'station_id', 'vehicle_id', 'customer_id', 'connector_type', 'session_status', 'tariff_id', 'firmware_ver', 'state_code', 'protocol_version']


In [0]:
# Cell 6 — Add Silver audit columns
# Same pattern as Day 8 — every Silver row gets lineage columns.

audited_df = (
    trimmed_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit("full"))
    .withColumn("silver_pipeline",    F.lit("pl_silver_blob_charging_sessions_v1"))
)

print("After adding audit columns:")
audited_df.printSchema()

After adding audit columns:
root
 |-- session_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- vehicle_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- plug_in_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- duration_min: integer (nullable = true)
 |-- energy_kwh: decimal(10,4) (nullable = true)
 |-- peak_kw: decimal(10,4) (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- tariff_id: string (nullable = true)
 |-- raw_device_temp_c: decimal(6,2) (nullable = true)
 |-- signal_strength_dbm: integer (nullable = true)
 |-- firmware_ver: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- protocol_version: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- updated_at: timestamp (nullable = false)
 |-- silver_ingested_at: timestamp (nullable = true)
 |--

In [0]:
# Cell 7 — Deduplicate on session_id
# Bronze may contain the same session_id across multiple hourly CSV files
# (e.g. an in-progress session updated across two hours).
# Keep the row with the most recent updated_at per session_id.

window = Window.partitionBy("session_id").orderBy(F.col("updated_at").desc())

deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

before = audited_df.count()
after  = deduped_df.count()
print(f"Before dedup : {before}")
print(f"After dedup  : {after}")
print(f"Duplicates removed : {before - after}")

Before dedup : 3010612
After dedup  : 1000000
Duplicates removed : 2010612


In [0]:
# Cell 8 — Write to Silver as Delta table (full overwrite)

(
    deduped_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Written to: {SILVER_PATH}")
print(f"Rows written: {deduped_df.count()}")

---------------------------------------------------------------------------
DateTimeException                         Traceback (most recent call last)
File <command-7590356060166501>, line 8
      1 # Cell 8 — Write to Silver as Delta table (full overwrite)
      3 (
      4     deduped_df.write
      5     .format("delta")
      6     .mode("overwrite")
      7     .option("overwriteSchema", "true")
----> 8     .save(SILVER_PATH)
      9 )
     11 print(f"Written to: {SILVER_PATH}")
     12 print(f"Rows written: {deduped_df.count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations
    705 )
    706 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/py

In [0]:
# Cell 9 — Verify Silver output

silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver charging_sessions schema:")
silver_df.printSchema()
print(f"\nTotal rows: {silver_df.count()}")
silver_df.show(5, truncate=True)

print("\nNull check on session_id (should be 0):")
print(silver_df.filter(F.col("session_id").isNull()).count())

print("\nStatus distribution:")
silver_df.groupBy("status").count().orderBy("count", ascending=False).show()